# 06 — Exportar os dados do painel (Fase 7)

Prepara a pasta **`app/data`**, que é tudo o que o painel Streamlit precisa para rodar sem o CSV de 470 MB:

- `xgboost_B.json`: o modelo XGBoost do cenário B em formato nativo (portável entre versões);
- `demo_amostra.parquet`: amostra do **teste** (todas as fraudes + parte das legítimas) com os scores dos cenários B e C e o peso que recompõe os números reais;
- `exemplos.json`: transações prontas para os botões do painel;
- `thresholds.json`, `meta.json` e os CSVs de resultados dos notebooks 04 e 05.

Runtime: **CPU**, leva 1 a 2 minutos. Pré-requisitos: notebooks 04 e 05 executados.

In [ ]:
import os, sys, time
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/fraud-detection-paysim'
except ImportError:
    PROJECT_DIR = os.getcwd()
os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))

import numpy as np, pandas as pd, joblib
import matplotlib
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

from fraud.config import load_config
cfg = load_config()
y = cfg['data']['target']
FIG = os.path.join(cfg['paths']['reports_dir'], 'figures'); os.makedirs(FIG, exist_ok=True)
MODELS = cfg['paths']['models_dir']; os.makedirs(MODELS, exist_ok=True)
SCORES = os.path.join(cfg['paths']['reports_dir'], 'scores'); os.makedirs(SCORES, exist_ok=True)
pd.options.display.float_format = '{:,.4f}'.format
print('Projeto:', PROJECT_DIR)

In [ ]:
import json, shutil
N_LEGIT_AMOSTRA = 30000
APP_DATA = os.path.join(PROJECT_DIR, 'app', 'data'); os.makedirs(APP_DATA, exist_ok=True)
FINAL = os.path.join(MODELS, 'final')
REPORTS = cfg['paths']['reports_dir']
for arq in [os.path.join(FINAL, 'xgboost_B.joblib'), os.path.join(FINAL, 'thresholds.json'),
            os.path.join(SCORES, 'test_xgboost_B_final.npy'), os.path.join(SCORES, 'test_xgboost_C_final.npy')]:
    assert os.path.exists(arq), f'Faltando {arq}: rode antes os notebooks 04 e 05.'
print('pré-requisitos ok')

## 1. Amostra de demonstração do teste

In [ ]:
from fraud.data import load_parquet
from fraud.split import make_split
from fraud.costs import legit_weight_for, REAL_FRAUD_RATE

df = load_parquet(cfg['paths']['parquet'])
_, _, test = make_split(df, cfg)
del df
test = test.reset_index(drop=True)
sB = np.load(os.path.join(SCORES, 'test_xgboost_B_final.npy')).astype('float64')
sC = np.load(os.path.join(SCORES, 'test_xgboost_C_final.npy')).astype('float64')
assert len(sB) == len(test) == len(sC)

rng = np.random.default_rng(cfg['seed'])
idx_f = np.flatnonzero(test[y].values == 1)
idx_l = np.flatnonzero(test[y].values == 0)
idx_l = rng.choice(idx_l, size=min(N_LEGIT_AMOSTRA, len(idx_l)), replace=False)
idx = np.sort(np.r_[idx_f, idx_l])

cols = ['step', 'type', 'amount', 'oldbalanceOrg', 'oldbalanceDest']
am = test.loc[idx, cols + [y]].copy()
am['score_B'], am['score_C'] = sB[idx], sC[idx]
n_leg_test = int((test[y] == 0).sum())
am['peso'] = np.where(am[y] == 1, 1.0, n_leg_test / len(idx_l))
am = am.reset_index(drop=True)
am.to_parquet(os.path.join(APP_DATA, 'demo_amostra.parquet'), index=False)

meta = {'n_teste': int(len(test)), 'n_fraudes_teste': int(len(idx_f)), 'n_legitimas_teste': n_leg_test,
        'n_amostra': int(len(am)), 'valor_fraudado_teste': float(test.loc[idx_f, 'amount'].sum()),
        'taxa_fraude_teste': float(test[y].mean()), 'taxa_fraude_real': REAL_FRAUD_RATE,
        'peso_legitima_taxa_real': float(legit_weight_for(test[y].values))}
json.dump(meta, open(os.path.join(APP_DATA, 'meta.json'), 'w', encoding='utf-8'), indent=2)
print(meta)

## 2. Modelo em formato JSON e conferência

In [ ]:
from fraud.scoring import load_model, score_transactions
modelo = joblib.load(os.path.join(FINAL, 'xgboost_B.joblib'))
arq_modelo = os.path.join(APP_DATA, 'xgboost_B.json')
modelo.save_model(arq_modelo)

m2 = load_model(arq_modelo)
conf = score_transactions(m2, am.head(2000))
dif = np.abs(conf - am['score_B'].head(2000).values).max()
print(f'diferença máxima entre o score do JSON e o score salvo: {dif:.2e}')
assert dif < 1e-4
print(f'modelo: {os.path.getsize(arq_modelo) / 1e6:.2f} MB')

## 3. Exemplos para os botões do painel

In [ ]:
am['score_B'] = score_transactions(m2, am)
fr, lg = am[am[y] == 1], am[am[y] == 0]

def perto(d, col, alvo):
    return d.iloc[(d[col] - alvo).abs().argsort().iloc[0]]

candidatos = [
    ('Fraude clássica', 'Transferência que esvazia a conta de origem: o padrão mais comum do PaySim.',
     perto(fr[(fr['type'] == 'TRANSFER') & (fr['score_B'] > 0.99)], 'amount', fr['amount'].median())),
    ('Fraude de alto valor', 'A maior fraude da amostra: o tipo de caso em que mais dinheiro está em jogo.',
     fr.sort_values('amount').iloc[-1]),
    ('Fraude difícil', 'A fraude com o menor score do modelo: mostra o limite da detecção.',
     fr.sort_values('score_B').iloc[0]),
    ('Pagamento comum', 'Compra típica, sem sinal de risco.',
     perto(lg[lg['type'] == 'PAYMENT'], 'amount', lg[lg['type'] == 'PAYMENT']['amount'].median())),
    ('Transferência grande', 'Valor alto, mas o modelo reconhece como normal.',
     lg[lg['type'].isin(['TRANSFER', 'CASH_OUT']) & (lg['score_B'] < 0.5)].sort_values('amount').iloc[-1]),
    ('Falso alarme', 'A transação legítima com o maior score: mostra um erro do modelo.',
     lg.sort_values('score_B').iloc[-1]),
]
t_auth_B = next(v for v in json.load(open(os.path.join(FINAL, 'thresholds.json'), encoding='utf-8'))['thresholds']['B'].values()
                if v['padrao'])['t_auth']
if t_auth_B is not None and lg['score_B'].max() < t_auth_B:
    candidatos[-1] = ('Legítima mais suspeita', 'A transação legítima com o maior score da amostra: mesmo ela fica '
                      'abaixo dos cortes, ou seja, nenhum falso alarme no cenário B.', candidatos[-1][2])
exemplos = [{'titulo': t, 'descricao': d, 'step': int(r['step']), 'type': str(r['type']),
             'amount': float(r['amount']), 'oldbalanceOrg': float(r['oldbalanceOrg']),
             'oldbalanceDest': float(r['oldbalanceDest']), 'eh_fraude': bool(r[y] == 1),
             'score': float(r['score_B'])} for t, d, r in candidatos]
json.dump(exemplos, open(os.path.join(APP_DATA, 'exemplos.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
display(pd.DataFrame(exemplos)[['titulo', 'type', 'amount', 'oldbalanceOrg', 'eh_fraude', 'score']])

## 4. Copiar cortes e resultados

In [ ]:
for origem, nome in [(os.path.join(FINAL, 'thresholds.json'), 'thresholds.json')] + \
        [(os.path.join(REPORTS, f), f) for f in ['final_5_modelos.csv', 'final_ensemble.csv',
                                                  'final_impacto_financeiro.csv', 'final_thresholds_3_niveis.csv']]:
    assert os.path.exists(origem), f'Faltando {origem}'
    shutil.copy(origem, os.path.join(APP_DATA, nome))
tot = 0
for f in sorted(os.listdir(APP_DATA)):
    tam = os.path.getsize(os.path.join(APP_DATA, f)); tot += tam
    print(f'{f:35s} {tam / 1e6:7.2f} MB')
print(f'TOTAL: {tot / 1e6:.2f} MB  ->  pasta {APP_DATA}')

## O que fazer depois
Baixe a pasta `app/data` do Drive e coloque-a em `app/data` no projeto do VS Code. Me avise o tamanho total impresso na última célula.